In [1]:
import sys
from pathlib import Path

root = Path(r"D:\Projects\speech-ai-learning")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.speech_dataset import SpeechDataset

dataset = SpeechDataset(
    manifest_path=root / "metadata" / "manifest.csv",
    audio_root=(
        root
        / "data"
        / "raw"
        / "common_voice_beijing26"
        / "zh-CN-beijing"
        / "clips"
    ),
    split="train",
    sample_rate=16000
)

print("Dataset样本数:",len(dataset))

sample = dataset[0]

print("字段:",list(sample.keys()))
print("UID:",sample["uid"])
print("音频类型:",type(sample["audio"]))
print("音频形状:", tuple(sample["audio"].shape))
print("采样点数:", sample["length"])
print("文本:", sample["text"])

assert sample["audio"].ndim == 1
assert sample["audio"].dtype.name if False else True
assert sample["length"] == sample["audio"].numel()
assert sample["length"] > 0

print("SpeechDataset最小测试通过")

Dataset样本数: 2281
字段: ['audio', 'length', 'uid', 'text']
UID: common_voice_zh-CN_27897382
音频类型: <class 'torch.Tensor'>
音频形状: (101376,)
采样点数: 101376
文本: 大厦的主要租户包括美林集团及美国运通大型金融机构。
SpeechDataset最小测试通过


In [1]:
#使用真实音频测试speech dataset
import sys
from pathlib import Path
import torch

root = Path(r"D:\Projects\speech-ai-learning")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.speech_dataset import SpeechDataset, collate_fn

dataset = SpeechDataset(
    manifest_path=root / "metadata" / "manifest.csv",
    audio_root=(
        root
        / "data"
        / "raw"
        / "common_voice_beijing26"
        / "zh-CN-beijing"
        / "clips"
    ),
    split="train",
    sample_rate=16000
)

print("Dataset样本数:", len(dataset))

#读取前20条样本
samples = [
    dataset[i]
    for i in range(min(20, len(dataset)))
]

#找到其中最短和最长的两条音频
short_sample = min(
    samples,
    key=lambda x: x["length"]
)

long_sample = max(
    samples,
    key=lambda x: x["length"],
)

print("短音频长度:", short_sample["length"])
print("长音频长度:", long_sample["length"])

#组成batch
batch = collate_fn([
    short_sample,
    long_sample
])

print("\nBatch结果")
print("audio shape:", batch["audio"].shape)
print("lengths:",batch["lengths"])
print("padding_mask shape:", batch["padding_mask"].shape)

print(
    "每条音频padding数量:",
    batch["padding_mask"].sum(dim=1)
)

print("UID:", batch["uid"])

Dataset样本数: 2281
短音频长度: 35136
长音频长度: 105408

Batch结果
audio shape: torch.Size([2, 105408])
lengths: tensor([ 35136, 105408])
padding_mask shape: torch.Size([2, 105408])
每条音频padding数量: tensor([70272,     0])
UID: ['common_voice_zh-CN_28374955', 'common_voice_zh-CN_28333336']


In [2]:
#验收检查
# 1. batch形状正确
assert batch["audio"].ndim == 2

#2.原始长度没有丢失
assert torch.equal(
    batch["lengths"],
    torch.tensor([
        short_sample["length"],
        long_sample["length"],
    ])
)

#3.mask与audio形状一致
assert (
    batch["padding_mask"].shape
    == batch["audio"].shape
)

#4.短音频有效区域不能被标记为padding
assert not batch["padding_mask"][0, :short_sample["length"]].any()

# 5. 短音频的填充区域必须全部为True
assert batch["padding_mask"][0, short_sample["length"]:].all()

# 6. 最长音频没有padding
assert not batch["padding_mask"][1].any()

# 7. padding位置的数值为0
assert torch.all(
    batch["audio"][batch["padding_mask"]] == 0
)

print("Collate全部验收通过")

Collate全部验收通过


In [3]:
#DataLoader冒烟测试
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

batch = next(iter(loader))

print("audio shape:", batch["audio"].shape)
print("lengths:", batch["lengths"])
print("padding_mask shape:", batch["padding_mask"].shape)
print("UID:", batch["uid"])

assert batch["audio"].shape[0] == 2
assert batch["lengths"].shape == (2,)
assert batch["padding_mask"].shape == batch["audio"].shape
assert len(batch["uid"]) == 2

print("DataLoader检查通过")

audio shape: torch.Size([2, 101376])
lengths: tensor([101376,  76608])
padding_mask shape: torch.Size([2, 101376])
UID: ['common_voice_zh-CN_27897382', 'common_voice_zh-CN_28304063']
DataLoader检查通过
